In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from joblib import dump
print("import done")

import done


In [5]:
# Load and preprocess the dataset
data = pd.read_csv(r"C:\Users\Hp\Desktop\BIA\DATA SCIENCE PROJECTS\ds_project_2\data\Twitter_Data.csv")
data.head()

,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative


In [13]:
# Convert 'clean_text' column to strings

data["clean_text"] = data["text"].astype(str)

data["clean_text"] = data["clean_text"].str.replace(
    r"[^a-zA-Z\s]", "", regex=True
).str.lower()

In [14]:
# Split data into training and testing sets
X = data['clean_text']
y = data['sentiment']

In [15]:
# Check unique values in the 'sentiment' column
unique_sentiments = y.unique()
print("Unique Sentiments:", unique_sentiments)

Unique Sentiments: <ArrowStringArray>
['neutral', 'negative', 'positive']
Length: 3, dtype: str


In [16]:
# Ensure that your labels are numeric
y = y.replace({'negative': 0, 'neutral': 1, 'positive': 2})

In [17]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
print(X_train.isnull().sum())
print(X_test.isnull().sum())

1
0


In [20]:
X_train = X_train.fillna("").astype(str)
X_test = X_test.fillna("").astype(str)

In [21]:
# Tokenize and pad text sequences

tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=100, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=100, padding='post')

In [22]:
# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [23]:
# One-hot encode labels
num_classes = len(unique_sentiments)
y_train_onehot = tf.keras.utils.to_categorical(y_train_encoded, num_classes=num_classes)
y_test_onehot = tf.keras.utils.to_categorical(y_test_encoded, num_classes=num_classes)

In [24]:
# Build a simple LSTM model with 3 output units
model = tf.keras.Sequential([
    Embedding(input_dim=5000, output_dim=100, input_length=100),
    LSTM(128),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

C:\Users\Hp\Desktop\BIA\DATA SCIENCE PROJECTS\ds_project_2\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [25]:
# Train the model with a reduced batch size
model.fit(X_train_pad, y_train_onehot, epochs=5, batch_size=32, validation_data=(X_test_pad, y_test_onehot))

Epoch 1/5
687/687 ━━━━━━━━━━━━━━━━━━━━ 62s 87ms/step - accuracy: 0.4019 - loss: 1.0892 - val_accuracy: 0.4057 - val_loss: 1.0868
Epoch 2/5
687/687 ━━━━━━━━━━━━━━━━━━━━ 57s 82ms/step - accuracy: 0.4042 - loss: 1.0875 - val_accuracy: 0.4057 - val_loss: 1.0880
Epoch 3/5
687/687 ━━━━━━━━━━━━━━━━━━━━ 59s 85ms/step - accuracy: 0.4043 - loss: 1.0876 - val_accuracy: 0.4057 - val_loss: 1.0868
Epoch 4/5
687/687 ━━━━━━━━━━━━━━━━━━━━ 61s 89ms/step - accuracy: 0.4043 - loss: 1.0873 - val_accuracy: 0.4057 - val_loss: 1.0867
Epoch 5/5
687/687 ━━━━━━━━━━━━━━━━━━━━ 61s 89ms/step - accuracy: 0.4039 - loss: 1.0874 - val_accuracy: 0.4057 - val_loss: 1.0867


In [26]:
# Save the trained model
model.save('sentiment_model.h5')
dump(tokenizer, 'tokenizer.joblib')

['tokenizer.joblib']